## Parte 4 — Ajuste fino y monitoreo 

### 4.a — ¿Hace falta ajuste fino? 

- Con tu `resultados.csv`, responde en **150–250 palabras**: ¿algo de tus fallos se arreglaría con ajuste fino y no con el prompt o con datos? Usa el árbol de la sesión 19 (mejor prompt → recuperar lo que falta → ajustar pesos) y ubica en él tus tres peores casos: qué falló y, por tanto, qué se toca. «No» es una respuesta válida si tus cifras la sostienen.

Respuesta: **NO**. 

Con el resultados_solver.csv de ambas versiones, recorrí el árbol de la sesión 19 (mejor prompt, recuperar información y ajustar pesos). Ninguno de mis tres peores casos justifica ajustar pesos.

En S2, el verificador pasó 2 de 12 comprobaciones. El programador devolvió contenido vacío por longitud y reintentó con 65 536 tokens, agotando la reserva del redactor e impidiendo generar informe.md. La v2 incorpora un freno de presupuesto, una solución de sistema, no de pesos. Sin embargo, el control también mejoró.

En P1, falló P107 porque faltaba enmascarar datos personales en traza.jsonl, lo que requiere un guardrail. En S1, falló S103 por una restricción incorrecta del verificador, corregida en el golden.

La versión mejorada aprobó 60 de 61 comprobaciones; solo falló P107. El ajuste fino tendría sentido si persistieran errores que ni el prompt ni la recuperación solucionan. Con apenas siete tareas, no hay evidencia suficiente para justificarlo. Por tanto, concluyo que no hace falta ajuste fino.


- Si decidieras ajustar, describe: qué **modelo de pesos abiertos** de la tabla (por su fila); qué datos y cuántos, y cómo los sacarías de tu golden set —y si sería ajuste supervisado (pares pregunta-respuesta) o por preferencias (pares aceptada/rechazada)—; qué **matrices adaptarías y con qué rango**, citando a Hu et al. (2021, «LoRA», Tabla 5 y §7.1) y Dettmers et al. (2023, «QLoRA», §4); y cómo sabrías que mejoró.

Respuesta: 

**Si ajustara (hipotético):** emplearía `open_weight_local` (llama3.1:8b), con ajuste supervisado de 60–100 pares subtarea-script aprobado del golden set. Aplicaría LoRA a todas las capas lineales, con rango \(r=8\). Evaluaría la mejora mediante la reducción de llamadas vacías en tareas reservadas, usando al menos tres corridas y comparando resultados frente al control.

### 4.b — Tres alertas



Define tres alertas de monitoreo (por ejemplo: latencia, costo o tokens por corrida, y tasa de bloqueos). Para cada una: umbral, **sacado de tus mediciones** (no de un manual), la ventana de tiempo en que se calcula, y qué mirarías primero en los spans si saltara. El umbral tiene que estar por encima de la resolución de tu golden set: con 15 casos, «más del 5 %» salta en el primer caso, porque cada caso vale 6,67 %.


**Resolución de mi golden set:** 7 tareas, así que una tarea vale 14,3 %; y 61 comprobaciones, así que una vale 1,6 %. Los umbrales están por encima de esa resolución.

**Alerta 1: tokens por corrida (costo).**
- **Umbral:** más de **450 000 tokens** en una corrida. Es el presupuesto de 500 000 menos la reserva de 50 000 del redactor, y sale de mi falla real: S2 (base) consumió 520 683 y se quedó sin informe, mientras que A, una corrida sana, consumió 81 239.
- **Ventana:** cada corrida, al terminar.
- **Primero en los spans:** el guardrail `presupuesto_tokens` (decisión y `techo_salida`) y las llamadas `llm` del programador con `fin=length`.

**Alerta 2: latencia por corrida.**
- **Umbral:** más de **<<máx. de corridas sanas + 25 %>> s**. Mis corridas sanas llegan a <<máx. sano>> s, y S2 fallida tardó 1 272 s.
- **Ventana:** cada corrida, y la mediana de cada pasada completa de las 7 tareas.
- **Primero en los spans:** el `agent` más largo (en S2, el programador: 836 s de 1 272) y cuántas veces repitió el crítico.

**Alerta 3: tasa de corridas fallidas o frenadas (bloqueos).**
- **Umbral:** **2 o más de 7 tareas (28,6 %)** con `status` fallido o con `presupuesto_agotado`. Una sola tarea (14,3 %) ya ocurrió en mi base (S2) y dispararía la alerta en cada pasada; «más del 5 %» saltaría con el primer caso.
- **Ventana:** una pasada completa del golden (las 7 tareas).
- **Primero en los spans:** el span `corrida` con error, luego el guardrail `presupuesto_tokens` y las decisiones de `traza.jsonl`.

Escogí las tres alertas para cubrir las tres cosas que fallaron o pudieron fallar en mis corridas: el costo, el tiempo y la tasa de fallos, y fijé cada umbral con una medición propia y no con un valor de manual. Primero consideré la resolución de mi golden set: con 7 tareas, una tarea vale 14,3 % y, con 61 comprobaciones, una comprobación vale 1,6 %, así que un umbral como «más del 5 %» saltaría con el primer caso y no distinguiría un fallo real del ruido. La alerta de tokens se fija en 450 000 por corrida, que es el presupuesto de 500 000 menos la reserva de 50 000 que protege al redactor: lo elegí porque mi peor caso, S2 en la base, consumió 520 683 tokens, agotó esa reserva y no entregó informe, mientras que una corrida sana como A consumió unos 81 000, de modo que el umbral queda lejos de lo normal y justo donde empieza el daño. La alerta de latencia se fija en el máximo de mis corridas sanas más un margen de 25 %, un margen que decidí yo para no saltar por variaciones normales del servidor compartido, y se calcula tanto por corrida como en la mediana de cada pasada completa del golden; la referencia de falla es S2, que tardó 1 272 s. La alerta de bloqueos y fallos salta con dos o más tareas de siete (28,6 %) con estado fallido o con el guardrail de presupuesto activado, porque una sola tarea (14,3 %) ya ocurrió en mi base y habría disparado la alerta en cada pasada. Para cada alerta, indico qué mirar primero en los spans: el guardrail `presupuesto_tokens` y las llamadas del programador cortadas por longitud en la de tokens, el agente más largo y las repeticiones del crítico en la de latencia, y el span `corrida` con error en la de fallos.

Los números salen de `scripts/umbrales_4b.py`, que lee los `spans.jsonl` exportados de Langfuse de la versión base y de la mejorada, agrupa los spans por traza (una traza es una corrida) y calcula para cada una su duración, tomada del span raíz `corrida`, sus tokens, sumando entrada y salida de los spans `llm`, y si falló, según el span raíz traiga un error. Con eso imprime la mediana, el percentil 95 y el máximo de duración y de tokens de las corridas sin error, y la fracción de corridas fallidas. Dejé fuera la corrida de control, porque sus caídas por `ConnectionResetError` son del servidor y no del solver, e inflarían la tasa de fallos. Los umbrales son provisionales: con 7 tareas y casi una corrida por tarea, no puedo estimar bien la variabilidad, y los revisaría con más pasadas del golden.